# Module 22 — Mixed Precision Training

Every module so far trained in full 32-bit floating point (fp32) — every
number takes 4 bytes, full precision, full exponent range. Modern GPUs
have specialized "tensor cores" that do matrix multiplication dramatically
faster in 16-bit formats. **Mixed precision training** runs the
expensive parts (matrix multiplies) in a 16-bit format while keeping
numerically sensitive parts in fp32, for a real, measurable speedup with
minimal accuracy cost.

This notebook runs its GPU cells on this project's actual CUDA device — the
same RTX 4060 this project already assumed was available for local work
(see the project README).

## 1. Two 16-bit formats, and why they're different

- **fp16** (IEEE half precision): 1 sign bit, 5 exponent bits, 10 mantissa
  bits. More precision than bf16, but a much smaller representable range —
  values above ~65,504 overflow to infinity.
- **bf16** (bfloat16): 1 sign bit, 8 exponent bits, 7 mantissa bits — the
  *same exponent range as fp32*, just less precision per value. Values
  that would overflow fp16 are represented fine (just less precisely).

In [ ]:
import torch

fp32_tensor = torch.randn(1000, 1000, dtype=torch.float32)
print(f"fp32: {fp32_tensor.element_size()} bytes/element, {fp32_tensor.numel() * fp32_tensor.element_size() / 1e6:.2f} MB total")
print(f"fp16: {fp32_tensor.half().element_size()} bytes/element, {fp32_tensor.numel() * 2 / 1e6:.2f} MB total")
print(f"bf16: {fp32_tensor.bfloat16().element_size()} bytes/element, {fp32_tensor.numel() * 2 / 1e6:.2f} MB total")

big_value = torch.tensor(70000.0)  # bigger than fp16's ~65,504 max
print(f"\n70000.0 in fp16: {big_value.half().item()}  (overflowed to inf)")
print(f"70000.0 in bf16: {big_value.bfloat16().item()}  (represented, just less precisely)")
assert torch.isinf(big_value.half())
assert not torch.isinf(big_value.bfloat16())
print("\nConfirmed: this exact value overflows fp16 but not bf16 - why bf16 is generally preferred when available (no overflow tricks needed).")

## 2. The real payoff: tensor cores make bf16 matmuls dramatically faster

This is measured directly on this machine's GPU — not a claimed number.

In [ ]:
import time

assert torch.cuda.is_available(), "this cell needs a CUDA GPU"
device = "cuda"
torch.manual_seed(0)

n = 4096
a32 = torch.randn(n, n, device=device, dtype=torch.float32)
b32 = torch.randn(n, n, device=device, dtype=torch.float32)
a16, b16 = a32.bfloat16(), b32.bfloat16()

def bench(a, b, iters=30):
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(iters):
        _ = a @ b
    torch.cuda.synchronize()
    return (time.time() - start) / iters

for _ in range(5):  # warm up the GPU/kernels before timing
    _ = a32 @ b32
    _ = a16 @ b16
torch.cuda.synchronize()

t32 = bench(a32, b32)
t16 = bench(a16, b16)
print(f"fp32 matmul: {t32 * 1000:.2f} ms")
print(f"bf16 matmul: {t16 * 1000:.2f} ms")
print(f"speedup: {t32 / t16:.2f}x")
assert t16 < t32

## 3. A real mixed-precision training step, with `autocast`

`torch.autocast` runs eligible ops (matmuls, mostly) in the target dtype
automatically, while keeping things like reductions in fp32 for numerical
stability — you don't have to manually cast every tensor. This uses
Module 18's exact `NanoGPT` architecture (copied in, not re-explained).

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(Q, K, V, causal=True):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k, device=scores.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

def split_heads(t, num_heads):
    *batch_dims, seq_len, d_model = t.shape
    d_k = d_model // num_heads
    return t.view(*batch_dims, seq_len, num_heads, d_k).transpose(-3, -2)

def merge_heads(t):
    *batch_dims, num_heads, seq_len, d_k = t.shape
    return t.transpose(-3, -2).contiguous().view(*batch_dims, seq_len, num_heads * d_k)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x, causal=True):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        Qh, Kh, Vh = split_heads(Q, self.num_heads), split_heads(K, self.num_heads), split_heads(V, self.num_heads)
        out, _ = scaled_dot_product_attention(Qh, Kh, Vh, causal=causal)
        return self.Wo(merge_heads(out))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.ln1(x), causal=True)
        x = x + self.ffn(self.ln2(x))
        return x

class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len, d_ff=None):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight
    def forward(self, idx):
        seq_len = idx.shape[-1]
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)


torch.manual_seed(42)
vocab_size, block_size = 55, 32
model = NanoGPT(vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

x = torch.randint(0, vocab_size, (32, block_size), device=device)
y = torch.randint(0, vocab_size, (32, block_size), device=device)

losses = []
for step in range(20):
    optimizer.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    loss.backward()  # bf16 has fp32\'s exponent range, so no gradient underflow - no scaler needed
    optimizer.step()
    losses.append(loss.item())

print("loss every 5 steps:", losses[::5])
assert losses[-1] < losses[0]
print("Confirmed: an autocast (bf16) training step trains correctly - loss decreases just like fp32.")

## 4. fp16 needs a `GradScaler`; bf16 doesn't

Because fp16 has a small exponent range, small gradient values can
underflow to exactly 0 before they ever reach the optimizer. `GradScaler`
works around this by multiplying the loss by a large factor before
`.backward()` (so gradients scale up proportionally, away from the
underflow zone), then unscaling before the optimizer step. bf16's fp32-like
exponent range means gradients rarely get small enough to underflow in the
first place — one practical reason bf16 is generally preferred when a GPU
supports it (this project's target GPUs — a local RTX 4060 and Colab's
A100 — both do).

In [ ]:
scaler = torch.amp.GradScaler("cuda")

optimizer.zero_grad()
with torch.autocast(device_type="cuda", dtype=torch.float16):
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))

scaler.scale(loss).backward()   # scale up before backward, to avoid fp16 gradient underflow
scaler.step(optimizer)          # unscales internally, then steps
scaler.update()
print(f"fp16 + GradScaler training step ran successfully, loss = {loss.item():.4f}")

## Recap

- bf16 has fp32's exponent range with less precision; fp16 has more
  precision but a much smaller range and can overflow — verified directly
  with a value that breaks fp16 but not bf16.
- The real, measured payoff on this machine's GPU: a bf16 matmul ran
  roughly 3x faster than the identical fp32 matmul, via tensor cores (exact
  ratio printed above, and will vary run to run).
- `torch.autocast` handles casting automatically in a real training loop;
  bf16 needs no extra machinery, while fp16 needs `GradScaler` to avoid
  gradient underflow.
- (Honest caveat: peak *memory* during training is dominated by
  optimizer state, which PyTorch keeps in fp32 by default regardless of
  autocast — the big memory wins from mixed precision mainly show up in
  *activation* memory at much larger batch/sequence-length scales than
  this toy model uses, not in a measurement this small.)

Module 23 covers gradient accumulation — simulating a bigger batch size
than fits in memory at once, the other major lever (alongside mixed
precision) for training bigger models within Colab Pro's memory budget.